<a href="https://colab.research.google.com/github/Drewbits/petrophysical-data-quality-workflow/blob/main/notebooks/01_Inventory_and_Explore_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Petrophysical Data Quality Workflow

## Notebook 1 – Load and Inspect LAS Files

### Objective

#This notebook demonstrates the first stage of a petrophysical data quality workflow.
#The goal is to load LAS files, inspect their metadata and curve information, and
#prepare the data for subsequent quality control, standardization, and normalization.

### Learning Objectives

#- Understand the structure of LAS files
#- Read well headers and metadata
#- Inspect curve mnemonics and units
#- Load log data into a pandas DataFrame
#- Verify data integrity before processing


# Install required packages
!pip install lasio pandas matplotlib numpy

In [ ]:
from pathlib import Path
import lasio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display DataFrames more cleanly
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
zip_path = Path("/content/drive/MyDrive/Datasets/15_9-19 A.zip")

print("ZIP exists:", zip_path.exists())
print("File name:", zip_path.name)
print("File size (MB):", round(zip_path.stat().st_size / 1_000_000, 2))

ZIP exists: True
File name: 15_9-19 A.zip
File size (MB): 66.62


In [ ]:
from zipfile import ZipFile

# Folder where the archive will be extracted
extract_folder = Path("/content/drive/MyDrive/Datasets")

# Create the folder if it doesn't exist
extract_folder.mkdir(exist_ok=True)

# Extract the ZIP archive
with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Extraction complete.")
print(f"Files extracted to: {extract_folder}")

Extraction complete.
Files extracted to: /content/drive/MyDrive/Datasets


In [ ]:
# Show the first few items in the extracted folder

for item in extract_folder.iterdir():
    print(item.name)

15_9-19 A


In [ ]:
# Find every file inside the extracted archive
all_files = [path for path in extract_folder.rglob("*") if path.is_file()]

print(f"Total files found: {len(all_files)}")

Total files found: 104


In [ ]:
inventory_df = pd.DataFrame(
    {
        "file_name": [path.name for path in all_files],
        "extension": [path.suffix.upper() if path.suffix else "[NO EXTENSION]" for path in all_files],
        "relative_path": [str(path.relative_to(extract_folder)) for path in all_files],
        "parent_folder": [path.parent.name for path in all_files],
        "size_mb": [path.stat().st_size / 1_000_000 for path in all_files],
    }
)

inventory_df["size_mb"] = inventory_df["size_mb"].round(3)

inventory_df.head(10)

,file_name,extension,relative_path,parent_folder,size_mb
0,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_PETROPHY...,15_9-F-9 A,0.429
1,WLC_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_COMPOSIT...,15_9-F-9 A,0.179
2,WLC_COMPOSITE_1_INF_1.PDF,.PDF,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_COMPOSIT...,15_9-F-9 A,0.075
3,WLC_PETROPHYSICAL_COMPOSITE_1_INF_1.PDF,.PDF,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_PETROPHY...,15_9-F-9 A,0.076
4,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_PETROPHYSI...,15_9-F-1,11.994
5,WLC_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_COMPOSITE_...,15_9-F-1,2.033
6,WLC_COMPOSITE_1_INF_1.PDF,.PDF,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_COMPOSITE_...,15_9-F-1,0.173
7,WLC_PETROPHYSICAL_COMPOSITE_1_INF_1.PDF,.PDF,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_PETROPHYSI...,15_9-F-1,0.201
8,WLC_COMPOSITE_2.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-15/WLC_COMPOSITE...,15_9-F-15,1.496
9,WLC_PETROPHYSICAL_COMPOSITE_2_INF_1.PDF,.PDF,15_9-19 A/04.COMPOSITE/15_9-F-15/WLC_PETROPHYS...,15_9-F-15,0.179


In [ ]:
file_type_summary = (
    inventory_df.groupby("extension")
    .agg(
        file_count=("file_name", "count"),
        total_size_mb=("size_mb", "sum"),
    )
    .sort_values("file_count", ascending=False)
    .reset_index()
)

file_type_summary["total_size_mb"] = file_type_summary["total_size_mb"].round(2)

file_type_summary

,extension,file_count,total_size_mb
0,.PDF,42,5.88
1,.DLIS,34,106.54
2,.ASC,9,0.16
3,.LIS,8,16.83
4,.TXT,3,0.00
5,.LTI,3,16.29
6,.LAS,2,4.20
7,.DB,1,0.00
8,.TIF,1,3.44
9,[NO EXTENSION],1,0.00


In [ ]:
log_extensions = [".LAS", ".DLIS", ".LIS", ".ASC"]

log_files_df = inventory_df[
    inventory_df["extension"].isin(log_extensions)
].copy()

print(f"Potential digital log files: {len(log_files_df)}")

log_files_df.head(20)

Potential digital log files: 53


,file_name,extension,relative_path,parent_folder,size_mb
0,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_PETROPHY...,15_9-F-9 A,0.429
1,WLC_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-9 A/WLC_COMPOSIT...,15_9-F-9 A,0.179
4,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_PETROPHYSI...,15_9-F-1,11.994
5,WLC_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-1/WLC_COMPOSITE_...,15_9-F-1,2.033
8,WLC_COMPOSITE_2.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-15/WLC_COMPOSITE...,15_9-F-15,1.496
11,WLC_PETROPHYSICAL_COMPOSITE_2.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-15/WLC_PETROPHYS...,15_9-F-15,6.488
12,WLC_PETROPHYSICAL_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-5/WLC_PETROPHYSI...,15_9-F-5,4.803
13,WLC_COMPOSITE_1.DLIS,.DLIS,15_9-19 A/04.COMPOSITE/15_9-F-5/WLC_COMPOSITE_...,15_9-F-5,1.391
16,WLC_PETROPHYSICAL_COMPOSITE_1.LIS,.LIS,15_9-19 A/04.COMPOSITE/15_9-F-7/WLC_PETROPHYSI...,15_9-F-7,0.525
18,WLC_PETROPHYSICAL_COMPOSITE.las,.LAS,15_9-19 A/04.COMPOSITE/15_9-F-7/WLC_PETROPHYSI...,15_9-F-7,1.581


I began by recursively inventorying the archive rather than assuming its contents. I captured file paths, formats, and sizes in a structured DataFrame, then summarized the archive by extension to determine which ingestion methods would be required.

In [ ]:
output_folder = Path("/content/drive/MyDrive/Datasets/volve_dataset_outputs")
output_folder.mkdir(exist_ok=True)

inventory_path = output_folder / "volve_file_inventory.csv"
summary_path = output_folder / "volve_file_type_summary.csv"

inventory_df.to_csv(inventory_path, index=False)
file_type_summary.to_csv(summary_path, index=False)

print(f"Saved: {inventory_path}")
print(f"Saved: {summary_path}")

Saved: /content/drive/MyDrive/Datasets/volve_dataset_outputs/volve_file_inventory.csv
Saved: /content/drive/MyDrive/Datasets/volve_dataset_outputs/volve_file_type_summary.csv
